# 02 · Does Combine Movement Carry Over for Defensive Backs?

**Working question:** Does the way a defensive back moves in the Combine's back-pedal-and-transition drill (how hard he brakes, how much speed he keeps, how fast he drives back out) show up in how he moves in NFL games, and does it predict who earns a starting role?

**Why this angle:** the 40-yard dash measures straight-line speed. Covering a receiver is mostly about stopping and changing direction. The Combine tracking data captures that directly at 10 frames per second, so I can measure it rather than rely on a stopwatch.

**Design note:** game tracking covers only the 510 cohort players, so most opposing receivers aren't tracked and defender-to-receiver separation can't be computed from the defender's side. Instead, I measure each defender's own in-game reaction after the ball is thrown.

**Plan**
1. Combine movement profile from the DB transition drill
2. In-game reaction burst after the throw
3. Career outcomes
4. Does tracking add information beyond standard Combine times?

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 60)
sns.set_theme(style="whitegrid")

CANDIDATES = [Path("/kaggle/input/nfl-big-data-bowl-2027"), Path("../data"), Path("data")]
DATA = next(p for p in CANDIDATES if (p / "players.csv").exists())
print("Reading from", DATA.resolve())

DB_POSITIONS = {"CB", "SS", "FS", "DB"}

In [ ]:
players = pd.read_csv(DATA / "players.csv")
combine = pd.read_csv(DATA / "combine_results.csv")
careers = pd.read_csv(DATA / "player_career_successes.csv")

db_ids = set(players.loc[players["nfl_position"].isin(DB_POSITIONS), "nfl_id"])
print(len(db_ids), "defensive backs")

## 1. Combine movement profile

From each back-pedal-and-transition attempt I take the speed trace and locate three moments: the back-pedal peak, the plant (slowest point between the two peaks), and the drive peak.

| Feature | Definition |
|---|---|
| `entry_speed` | peak speed while back-pedaling |
| `speed_retained` | speed at the plant ÷ entry speed |
| `peak_decel` | hardest braking into the plant (yd/s²) |
| `reaccel_time` | seconds from the plant to 80% of drive-out peak speed |
| `drive_speed` | peak speed after the transition |

Acceleration is recomputed as the signed change in speed, so braking and accelerating can be told apart. A player's profile is the median across his attempts.

In [ ]:
ct = pd.read_csv(
    DATA / "combine_tracking.csv",
    usecols=["nfl_id", "event_id", "attempt", "entity_type", "time", "drill_type", "drill_name", "s", "dir"],
)
ct = ct[(ct["entity_type"] == "PLAYER") & (ct["drill_type"] == "SKILL_DRILLS_DB")]
print(ct["drill_name"].value_counts())

In [ ]:
transition = ct[ct["drill_name"].str.contains("BACK_PEDAL_AND_TRANSITION", na=False)].copy()
transition["time"] = pd.to_datetime(transition["time"], format="ISO8601")
transition = transition.sort_values(["event_id", "attempt", "time"])

def attempt_features(g):
    t = (g["time"] - g["time"].iloc[0]).dt.total_seconds().to_numpy()
    s = g["s"].to_numpy()
    if len(s) < 10:
        return None
    a = np.gradient(s, t)
    half = len(s) // 2
    i1 = int(np.argmax(s[:half]))                  # back-pedal peak
    i2 = half + int(np.argmax(s[half:]))           # drive-out peak
    if i2 - i1 < 3:
        return None
    im = i1 + int(np.argmin(s[i1:i2 + 1]))         # plant
    after = np.where(s[im:i2 + 1] >= 0.8 * s[i2])[0]
    return pd.Series({
        "entry_speed": s[i1],
        "speed_retained": s[im] / s[i1] if s[i1] > 0 else np.nan,
        "peak_decel": -a[i1:im + 1].min(),
        "reaccel_time": t[im + after[0]] - t[im] if len(after) else np.nan,
        "drive_speed": s[i2],
    })

attempts = (transition.groupby(["nfl_id", "event_id", "attempt"])
                      .apply(attempt_features).dropna(how="all").reset_index())
combine_moves = attempts.groupby("nfl_id")[["entry_speed", "speed_retained", "peak_decel",
                                            "reaccel_time", "drive_speed"]].median()
print(f"{len(combine_moves)} players with a movement profile")
combine_moves.describe().round(2)

Sanity check: speed traces for a few attempts with the detected plant marked.

In [ ]:
sample = attempts.sample(6, random_state=1)
fig, axes = plt.subplots(2, 3, figsize=(14, 6), sharey=True)
for ax, (_, r) in zip(axes.flat, sample.iterrows()):
    g = transition[(transition["event_id"] == r["event_id"]) & (transition["attempt"] == r["attempt"])]
    t = (g["time"] - g["time"].iloc[0]).dt.total_seconds()
    ax.plot(t, g["s"])
    ax.set_title(f"player {r['nfl_id']}, attempt {r['attempt']}", fontsize=9)
    ax.set_xlabel("seconds"); ax.set_ylabel("speed (yd/s)")
plt.tight_layout()

## 2. In-game reaction burst

For each pass play a defensive back is on the field for, I look at the first second after the ball is thrown (`pass_forward`):

- `burst_accel`: peak acceleration in that window
- `closing_yards`: distance covered in that window

These are averaged per player over plays where he had a coverage assignment. Tracking files are large, so they're read in chunks and filtered to defensive backs.

In [ ]:
WINDOW = 1.0  # seconds after the throw
usecols = ["game_id", "play_id", "nfl_id", "time", "s", "dis", "event"]

def play_bursts(frames):
    out = []
    for (gid, pid, nid), g in frames.groupby(["game_id", "play_id", "nfl_id"], sort=False):
        thrown = g.index[g["event"] == "pass_forward"]
        if len(thrown) == 0:
            continue
        t0 = g.at[thrown[0], "time"]
        w = g[(g["time"] >= t0) & (g["time"] <= t0 + pd.Timedelta(seconds=WINDOW))]
        if len(w) < 5:
            continue
        t = (w["time"] - t0).dt.total_seconds().to_numpy()
        out.append((gid, pid, nid, np.gradient(w["s"].to_numpy(), t).max(), w["dis"].iloc[1:].sum()))
    return out

rows = []
for season in (2023, 2024, 2025):
    kept = [c[c["nfl_id"].isin(db_ids)] for c in
            pd.read_csv(DATA / f"game_tracking_{season}.csv", usecols=usecols, chunksize=2_000_000)]
    frames = pd.concat(kept, ignore_index=True)
    frames["time"] = pd.to_datetime(frames["time"], format="ISO8601")
    frames = frames.sort_values(["game_id", "play_id", "nfl_id", "time"]).reset_index(drop=True)
    rows += play_bursts(frames)
    print(season, f"{len(rows):,} DB pass plays so far")

bursts = pd.DataFrame(rows, columns=["game_id", "play_id", "nfl_id", "burst_accel", "closing_yards"])

In [ ]:
pp = pd.read_csv(DATA / "player_play.csv",
                 usecols=["game_id", "play_id", "nfl_id", "coverage_assignment", "team_coverage_man_zone",
                          "play_nullified_by_penalty"])
bursts = bursts.merge(pp, on=["game_id", "play_id", "nfl_id"], how="left")
bursts = bursts[bursts["coverage_assignment"].notna() & (bursts["play_nullified_by_penalty"] != "Y")]

MIN_PLAYS = 30
game_moves = (bursts.groupby("nfl_id")
                    .agg(burst_accel=("burst_accel", "mean"), closing_yards=("closing_yards", "mean"),
                         man_share=("team_coverage_man_zone", lambda s: (s == "MAN_COVERAGE").mean()),
                         coverage_plays=("burst_accel", "size"))
                    .query("coverage_plays >= @MIN_PLAYS"))
print(f"{len(game_moves)} DBs with at least {MIN_PLAYS} coverage plays")
game_moves.describe().round(2)

## 3. Bringing it together

In [ ]:
analysis = (combine_moves
            .join(combine.set_index("nfl_id")[["forty", "ten_yd_split", "three_cone", "short_shuttle",
                                               "combine_weight", "ngs_athleticism_score"]])
            .join(game_moves, how="left")
            .join(careers.set_index("nfl_id")[["career_games_started", "career_games_active",
                                               "career_defensive_snaps"]])
            .join(players.set_index("nfl_id")[["display_name", "nfl_position", "draft_overall_pick"]]))
analysis["start_rate"] = analysis["career_games_started"] / analysis["career_games_active"].replace(0, np.nan)

movement = ["entry_speed", "speed_retained", "peak_decel", "reaccel_time", "drive_speed"]
standard = ["forty", "ten_yd_split", "three_cone", "short_shuttle"]
outcomes = ["burst_accel", "closing_yards", "start_rate", "career_defensive_snaps"]

plt.figure(figsize=(9, 6))
sns.heatmap(analysis[movement + standard + outcomes].corr(method="spearman")
            .loc[movement + standard, outcomes], annot=True, fmt=".2f", cmap="vlag", center=0)
plt.title("Spearman correlation: Combine measures vs. NFL outcomes")

## 4. Does tracking add information beyond the stopwatch?

For each outcome, compare cross-validated R² for:
- **Standard:** 40-yard dash, 10-yard split, 3-cone, short shuttle
- **Standard + movement:** the same plus the transition-drill features

Combine classes are small, so I use repeated K-fold cross-validation and report the spread, not a single split.

In [ ]:
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import RepeatedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def cv_r2(data, features, target):
    d = data.dropna(subset=[target])
    model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 20)))
    scores = cross_val_score(model, d[features], d[target],
                             cv=RepeatedKFold(n_splits=5, n_repeats=10, random_state=0), scoring="r2")
    return scores.mean(), scores.std(), len(d)

results = []
for target in outcomes:
    for label, feats in {"standard": standard, "standard + movement": standard + movement}.items():
        m, sd, n = cv_r2(analysis, feats, target)
        results.append({"outcome": target, "features": label, "mean_r2": m, "sd_r2": sd, "n": n})
pd.DataFrame(results).round(3)

## Next steps

- Refine the plant detection after reviewing more speed traces
- Split the in-game burst by man vs. zone coverage, where reaction demands differ
- Look at players whose movement profile and draft position disagree most
- Animate representative Combine attempts and game plays side by side